# Experiment 50 v2 — Overlap-aware relevance bootstrap and BH-FDR

This controller reuses the trained rerankers and cached long-horizon evaluation data from Experiment 39. It creates non-destructive copies of the two relevant notebooks and changes only the inferential layer:

- block length: `ceil(horizon / dataset window stride) + 1`;
- horizons: 96, 192, 336, and 720;
- primary family: Learned versus Pattern, AnalogFutureMSE, across 6 datasets × 4 horizons = 24 tests;
- multiplicity: Benjamini–Hochberg FDR at `q=0.05`.

No reranker architecture, target, split, candidate pool, or test prediction is changed. Existing checkpoints and caches are loaded with `FORCE_RETRAIN=False`.

In [ ]:
from pathlib import Path
import copy, json, math, os, re, subprocess, sys, time
import numpy as np
import pandas as pd

WORK = Path("/data/dataset/strong_forecaster/predictive_relevance_long_horizon")
PATCHED = WORK / "patched_notebooks"
EXECUTED = WORK / "executed_notebooks_dynamic_block"
OUT = WORK / "overlap_aware_bootstrap_fdr"
EXECUTED.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

KERNEL = "stock_regime"
TIMEOUT = 7200
Q = 0.05

DATA_ROOT_CANDIDATES = [
    Path("/data/pcw_workspace/Time-Series-Library/dataset"),
    Path("/data/Time-Series-Library/dataset"),
    Path("/data/code/Time-Series-Library/dataset"),
]
REQUIRED_DATA = [
    "ETT-small/ETTh1.csv", "weather/weather.csv",
    "electricity/electricity.csv", "traffic/traffic.csv",
    "exchange_rate/exchange_rate.csv", "Solar/solar_AL.txt",
]
DATA_ROOT = next(
    (p for p in DATA_ROOT_CANDIDATES if all((p / r).is_file() for r in REQUIRED_DATA)),
    None,
)
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not locate a dataset root containing all six required files: "
        + str(DATA_ROOT_CANDIDATES)
    )

REPO_ROOT = Path("/data/code/which-histories-matter")
if not REPO_ROOT.is_dir():
    raise FileNotFoundError(REPO_ROOT)

print("Work root:", WORK)
print("Output:", OUT)
print("Repository root:", REPO_ROOT)
print("Dataset root:", DATA_ROOT)

## 1. Locate Experiment 39 notebooks

In [ ]:
def find_one(pattern):
    hits = sorted(PATCHED.glob(pattern))
    if len(hits) != 1:
        raise RuntimeError(f"Expected one {pattern} under {PATCHED}, found {hits}")
    return hits[0]

SOURCES = {
    "base": find_one("00_cross_domain_base_longH.ipynb"),
    "confirmatory": find_one("confirmatory_benchmark_longH.ipynb"),
}

for k,p in SOURCES.items():
    print(k, p)
print("PASS: Experiment 39 patched notebooks found.")

## 2. Patch only block length and output filename

In [ ]:
TARGETS = {}

def patch_notebook(kind, src):
    nb = json.loads(src.read_text(encoding="utf-8"))
    replaced_block = 0
    replaced_output = 0
    for cell in nb["cells"]:
        if cell.get("cell_type") != "code":
            continue
        text = "".join(cell.get("source", []))
        # Use the same 5,000 replicates for both notebook families.
        text = text.replace("N_BOOT = 3000", "N_BOOT = 5000")
        # Both keyword and positional call styles occur in the public notebooks.
        new = text.replace(
            "block_len=BLOCK_ANCHORS,",
            "block_len=(math.ceil(H / WINDOW_STRIDE[dataset_name]) + 1),",
        )
        new = new.replace(
            "                    BLOCK_ANCHORS,\n                    N_BOOT,",
            "                    (math.ceil(H / WINDOW_STRIDE[dataset_name]) + 1),\n                    N_BOOT,",
        )
        replaced_block += (new != text)
        text = new
        if kind == "base":
            new = text.replace('"08_moving_block_bootstrap.csv"', '"08_moving_block_bootstrap_overlap_aware.csv"')
        else:
            new = text.replace('"12_moving_block_bootstrap.csv"', '"12_moving_block_bootstrap_overlap_aware.csv"')
        replaced_output += (new != text)
        cell["source"] = new.splitlines(keepends=True)
    if replaced_block == 0 or replaced_output == 0:
        raise RuntimeError(f"Patch failed for {kind}: block={replaced_block}, output={replaced_output}")
    nb.setdefault("metadata", {})["kernelspec"] = {
        "display_name":"Python 3.12 (Stock Regime)", "language":"python", "name":"stock_regime"
    }
    dst = OUT / f"{kind}_overlap_aware.ipynb"
    dst.write_text(json.dumps(nb, ensure_ascii=False, indent=1), encoding="utf-8")
    return dst, replaced_block, replaced_output

for kind, src in SOURCES.items():
    TARGETS[kind] = patch_notebook(kind, src)[0]
    print("PATCHED", kind, "->", TARGETS[kind])

print("PASS: only inferential block calls and bootstrap output names were patched.")

## 3. Execute with cached windows and trained models

The first execution can still spend time reconstructing query-level evaluation tables, especially for Traffic and Solar, but it should not retrain models when Experiment 39 artifacts are present.

In [ ]:
execution=[]
env=os.environ.copy()
env["WHM_KERNEL_NAME"] = KERNEL
env["WHM_REPO_ROOT"] = str(REPO_ROOT)
env["WHM_DATA_ROOT"] = str(DATA_ROOT)
env["WHM_WORK_ROOT"] = str(WORK)

for kind, src in TARGETS.items():
    dst = EXECUTED / f"{kind}_overlap_aware_executed.ipynb"
    cmd=[
        sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(src),
        "--output",str(dst),
        "--ExecutePreprocessor.kernel_name="+KERNEL,
        "--ExecutePreprocessor.timeout="+str(TIMEOUT),
    ]
    print("EXECUTING",kind)
    t=time.time()
    proc=subprocess.run(cmd,cwd=str(src.parent),env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    mins=(time.time()-t)/60
    print(proc.stdout[-6000:])
    execution.append({"Notebook":kind,"ReturnCode":proc.returncode,"Minutes":mins,"ExecutedPath":str(dst)})
    if proc.returncode != 0:
        raise RuntimeError(f"{kind} failed with return code {proc.returncode}")

execution_df=pd.DataFrame(execution)
execution_df.to_csv(OUT/"00_execution_log.csv",index=False)
display(execution_df)
print("PASS: both overlap-aware relevance notebooks executed.")

## 4. Assemble the primary 24-test family and apply BH-FDR

In [ ]:
def locate_result(filename):
    hits=sorted(WORK.rglob(filename))
    if len(hits) != 1:
        raise RuntimeError(f"Expected one {filename}, found {hits}")
    return hits[0]

paths={
    "base":locate_result("08_moving_block_bootstrap_overlap_aware.csv"),
    "confirmatory":locate_result("12_moving_block_bootstrap_overlap_aware.csv"),
}
frames=[]
for source,p in paths.items():
    d=pd.read_csv(p)
    d["SourceNotebook"]=source
    d["SourcePath"]=str(p)
    frames.append(d)
all_boot=pd.concat(frames,ignore_index=True)

primary=all_boot[
    (all_boot["Baseline"]=="Pattern") &
    (all_boot["Proposed"]=="Learned") &
    (all_boot["Metric"]=="AnalogFutureMSE")
].copy()
primary=primary.drop_duplicates(["Dataset","Horizon","Baseline","Proposed","Metric"])
primary=primary.sort_values(["Dataset","Horizon"]).reset_index(drop=True)

if len(primary) != 24 or primary["Dataset"].nunique()!=6:
    display(primary)
    raise RuntimeError(f"Expected 24 primary conditions across 6 datasets, found {len(primary)}")

# P_gt_0 is the fraction of bootstrap mean improvements above zero.
# The finite-replicate floor prevents an estimated p-value of exactly zero.
nboot=np.full(len(primary),5000)
tail=np.minimum(primary["P_gt_0"],1-primary["P_gt_0"])
primary["P_two_sided"]=2*np.maximum(tail,1/(nboot+1))

def bh(p):
    p=np.asarray(p,float);m=len(p);order=np.argsort(p);ranked=p[order]
    adj=np.minimum.accumulate((ranked*m/np.arange(1,m+1))[::-1])[::-1]
    out=np.empty(m);out[order]=np.clip(adj,0,1);return out

primary["P_BH_24"]=bh(primary["P_two_sided"])
primary["FDR_Significant"]=primary["P_BH_24"]<=Q
primary["Direction"]=np.where(primary["ObservedImprovement"]>0,"improve",np.where(primary["ObservedImprovement"]<0,"degrade","tie"))
primary["FDR_Class"]=np.where(primary["FDR_Significant"],primary["Direction"],"not significant")
primary.to_csv(OUT/"01_relevance_overlap_aware_bh_fdr_24.csv",index=False)

print("PASS: exact 24-condition primary relevance family completed.")
display(primary)

## 5. Paper-ready summaries

In [ ]:
overall=pd.DataFrame([{
    "Conditions":len(primary),
    "LearnedBeatsPattern":int((primary.ObservedImprovement>0).sum()),
    "RawSignificant":int((primary.P_two_sided<0.05).sum()),
    "FDRSignificant":int(primary.FDR_Significant.sum()),
    "FDRImprovements":int(((primary.FDR_Significant)&(primary.Direction=="improve")).sum()),
    "FDRDegradations":int(((primary.FDR_Significant)&(primary.Direction=="degrade")).sum()),
}])
by_dataset=primary.groupby("Dataset",as_index=False).agg(
    Conditions=("Horizon","size"),
    PositiveEffects=("ObservedImprovement",lambda x:int((x>0).sum())),
    FDRSignificant=("FDR_Significant","sum"),
)
overall.to_csv(OUT/"02_relevance_fdr_overall.csv",index=False)
by_dataset.to_csv(OUT/"03_relevance_fdr_by_dataset.csv",index=False)
primary[["Dataset","Horizon","ObservedImprovement","CI_2.5%","CI_97.5%","P_two_sided","P_BH_24","FDR_Class"]].to_csv(
    OUT/"04_relevance_paper_ready.csv",index=False
)
display(overall)
display(by_dataset)
print("Output files:")
for p in sorted(OUT.glob("*.csv")): print(" -",p)